In [5]:
from torch_geometric.nn import MessagePassing
import torch

In [3]:
class myGAT(MessagePassing):

    def __init__(self, in_channels, out_channels, heads = 2,
                 negative_slope = 0.2, dropout = 0., **kwargs):
        super(myGAT, self).__init__(node_dim=0, **kwargs)

        self.in_channels = in_channels 
        self.out_channels = out_channels 
        self.heads = heads 
        self.negative_slope = negative_slope
        self.dropout = dropout

        self.lin_l = None
        self.lin_r = None
        self.att_l = None
        self.att_r = None
     
        
        self.lin_l = Linear(in_channels, heads*out_channels)
        self.lin_r = self.lin_l

        self.att_l = Parameter(torch.Tensor(1, heads, out_channels).float())
        self.att_r = Parameter(torch.Tensor(1, heads, out_channels).float())

        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.lin_l.weight)
        nn.init.xavier_uniform_(self.lin_r.weight)
        nn.init.xavier_uniform_(self.att_l)
        nn.init.xavier_uniform_(self.att_r)

    def forward(self, x, edge_index, size = None):
        
        H, C = self.heads, self.out_channels 

        
        x_source = self.lin_l(x).view(-1,H,C) 
        x_target = self.lin_r(x).view(-1,H,C) 

        
        alpha_l = (x_source * self.att_l).sum(dim=-1) 
        alpha_r = (x_target * self.att_r).sum(dim=-1) 

        
        out = self.propagate(edge_index, x=(x_source, x_target), alpha=(alpha_l, alpha_r),size=size) 
        out = out.view(-1, self.heads * self.out_channels) 

        return out

    def message(self, x_j, alpha_j, alpha_i, index, ptr, size_i):
        
        attention = F.leaky_relu((alpha_j + alpha_i), self.negative_slope) 
        attention = softmax(attention, index, ptr, size_i) 
        attention = F.dropout(attention, p=self.dropout, training=self.training) 

        
        out = x_j * attention.unsqueeze(-1)  

        return out

    def aggregate(self, inputs, index, dim_size = None):
      
        out = torch_scatter.scatter(inputs, index, dim=self.node_dim, 
                                    dim_size=dim_size, reduce='sum')
  
        return out

In [6]:
class GATmodif(torch.nn.Module):
    def __init__(self,input_dim, hidden_dim, output_dim,args):
        super(GATmodif, self).__init__()

        self.conv1 = myGAT(input_dim, hidden_dim)
        self.conv2 = myGAT(args['heads'] * hidden_dim, hidden_dim)
       
        self.conv1 = GATConv(input_dim, hidden_dim, heads=args['heads'])
        self.conv2 = GATConv(args['heads'] * hidden_dim, hidden_dim, heads=args['heads'])
      

        self.post_mp = nn.Sequential(
            nn.Linear(args['heads'] * hidden_dim, hidden_dim), nn.Dropout(args['dropout'] ), 
            nn.Linear(hidden_dim, output_dim))
        
    def forward(self, data, adj=None):
        x, edge_index = data.x, data.edge_index
 
        x = self.conv1(x, edge_index)
        x = F.dropout(F.relu(x), p=0.5, training=self.training)

 
        x = self.conv2(x, edge_index)
        x = F.dropout(F.relu(x), p=0.5, training=self.training)

    
        x = self.post_mp(x)
        return F.sigmoid(x)

In [7]:
class GnnTrainer(object):
  
  def __init__(self, model):
    self.model = model
    self.metric_manager = MetricManager(modes=["train", "val"])

  def train(self, data_train, optimizer, criterion, scheduler, args):
  
    self.data_train = data_train
    for epoch in range(args['epochs']):
        self.model.train()
        optimizer.zero_grad()
        out = self.model(data_train)

        out = out.reshape((data_train.x.shape[0]))
        loss = criterion(out[data_train.train_idx], data_train.y[data_train.train_idx])
 
        target_labels = data_train.y.detach().cpu().numpy()[data_train.train_idx]
        pred_scores = out.detach().cpu().numpy()[data_train.train_idx]
        train_acc, train_f1,train_f1macro, train_aucroc, train_recall, train_precision, train_cm = self.metric_manager.store_metrics("train", pred_scores, target_labels)


  
        loss.backward()
        optimizer.step()

  
        self.model.eval()
        target_labels = data_train.y.detach().cpu().numpy()[data_train.valid_idx]
        pred_scores = out.detach().cpu().numpy()[data_train.valid_idx]
        val_acc, val_f1,val_f1macro, val_aucroc, val_recall, val_precision, val_cm = self.metric_manager.store_metrics("val", pred_scores, target_labels)

        if epoch%5 == 0:
          print("epoch: {} - loss: {:.4f} - accuracy train: {:.4f} -accuracy valid: {:.4f}  - val roc: {:.4f}  - val f1micro: {:.4f}".format(epoch, loss.item(), train_acc, val_acc, val_aucroc,val_f1))


  def predict(self, data=None, unclassified_only=True, threshold=0.5):

    self.model.eval()
    if data is not None:
      self.data_train = data

    out = self.model(self.data_train)
    out = out.reshape((self.data_train.x.shape[0]))

    if unclassified_only:
      pred_scores = out.detach().cpu().numpy()[self.data_train.test_idx]
    else:
      pred_scores = out.detach().cpu().numpy()

    pred_labels = pred_scores > threshold

    return {"pred_scores":pred_scores, "pred_labels":pred_labels}


  def save_metrics(self, save_name, path="./save/"):
    file_to_store = open(path + save_name, "wb")
    pickle.dump(self.metric_manager, file_to_store)
    file_to_store.close()

  def save_model(self, save_name, path="./save/"):
    torch.save(self.model.state_dict(), path + save_name)

In [8]:
class MetricManager(object):
  def __init__(self, modes=["train", "val"]):

    self.output = {}

    for mode in modes:
      self.output[mode] = {}
      self.output[mode]["accuracy"] = []
      self.output[mode]["f1micro"] = []
      self.output[mode]["f1macro"] = []
      self.output[mode]["aucroc"] = []
      #new
      self.output[mode]["precision"] = []
      self.output[mode]["recall"] = []
      self.output[mode]["cm"] = []

  def store_metrics(self, mode, pred_scores, target_labels, threshold=0.5):


    pred_labels = pred_scores > threshold
    accuracy = accuracy_score(target_labels, pred_labels)
    f1micro = f1_score(target_labels, pred_labels,average='micro')
    f1macro = f1_score(target_labels, pred_labels,average='macro')
    aucroc = roc_auc_score(target_labels, pred_scores)

    recall = recall_score(target_labels, pred_labels)
    precision = precision_score(target_labels, pred_labels)
    cm = confusion_matrix(target_labels, pred_labels)


    self.output[mode]["accuracy"].append(accuracy)
    self.output[mode]["f1micro"].append(f1micro)
    self.output[mode]["f1macro"].append(f1macro)
    self.output[mode]["aucroc"].append(aucroc)
    #new
    self.output[mode]["recall"].append(recall)
    self.output[mode]["precision"].append(precision)
    self.output[mode]["cm"].append(cm)
    
    return accuracy, f1micro,f1macro, aucroc,recall,precision,cm
  

  def get_best(self, metric, mode="val"):


    best_results = {}
    i = np.array(self.output[mode][metric]).argmax()


    for m in self.output[mode].keys():
      best_results[m] = self.output[mode][m][i]
    
    return best_results

In [ ]:

args={"epochs":10, 'lr':0.01, 'weight_decay':1e-5, 'prebuild':False, 'heads':2, 'num_layers': 2, 'hidden_dim': 128, 'dropout': 0.5 }
model = GATmodif(data_train.num_node_features, args['hidden_dim'], 1, args) # Change model as required, but arguments are consistent


data_train = data_train.to(device)


optimizer = torch.optim.Adam(model.parameters(), lr=args['lr'], weight_decay=args['weight_decay'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min')
criterion = torch.nn.BCELoss()


gnn_trainer_gatmodif = GnnTrainer(model)
gnn_trainer_gatmodif.train(data_train, optimizer, criterion, scheduler, args)
gnn_trainer_gatmodif.save_metrics("GATmodifhead2_newmetrics.results", path=FOLDERNAME + "/save_results/")
gnn_trainer_gatmodif.save_model("GATmodifhead2_newmetrics.pth", path=FOLDERNAME + "/save_results/")